# YOLO26-M (Ultralytics) — Training

Fine-tunes the COCO-pretrained `yolo26m.pt` on Construction-PPE. The test-set evaluation is in
[`yolo26m_eval.ipynb`](yolo26m_eval.ipynb).

| Setting | Value |
|---|---|
| Input size | 640×640 |
| Optimizer, LR schedule, loss | Ultralytics defaults for this model (`optimizer="auto"`) |
| Batch size | 16 |
| Epochs | `EPOCH_BUDGET` = 10 (shared scratch-run budget) |
| Checkpoint | best validation fitness → `models/yolo26m/weights/best.pt` |

> **Scratch run.** The shared `EPOCH_BUDGET` (`experiment.py`) is **10** so the notebook runs end to end quickly. For
> the actual runs, change this to early stopping: pass a larger `epochs` budget plus `patience=<N>` to `model.train`, which stops once validation fitness has not improved for N epochs.

**Data augmentation** is shared by all four models (`ppe_detection_evaluation.dataset.augmentations`). `SharedAugmentation` reseeds
from `(SEED, epoch, image name)`, so every model trains on exactly the same examples; only the order within an epoch
differs.

| # | Augmentation | Parameters | p |
|---|---|---|---|
| A1 | Horizontal flip | — | 0.5 |
| A2 | Scale + translate | scale 0.75–1.25, translate ±10 %, gray fill 114 | 1.0 |
| A3 | Colour jitter | brightness ±0.4, contrast ±0.3, saturation ±0.5, hue ±0.015 | 1.0 |
| A4 | One of blur / noise / JPEG | σ 0.1–1.5; σ 0.02–0.06; quality 40–90 | 0.25 |

Every image is first resized to 640×640. Boxes keeping less than 40 % of their area, or narrower than 3 px, are dropped.
Validation and test images are only resized.

Ultralytics' own augmentation (mosaic, HSV, perspective, flips and its built-in Albumentations) is **bypassed**:
`SharedAugmentationDetectionTrainer` trains on a dataset whose `__getitem__` calls the same `load_training_example` as the torchvision models,
followed only by Ultralytics' eval-mode letterbox (a no-op at 640×640) and `Format`. The built-in augmentation arguments
are also zeroed (`DISABLE_BUILTIN_AUGMENTATION` in `ultralytics_adapter.py`), so the run's `args.yaml` records it.

In [ ]:
import csv
import os

%matplotlib inline
import matplotlib.pyplot as plt
from ultralytics import YOLO

from ppe_detection_evaluation.dataset.augmentations import IMAGE_SIZE, SharedAugmentation
from ppe_detection_evaluation.dataset.core import check_dataset, class_box_stats, download_dataset, verify_same_examples
from ppe_detection_evaluation.dataset.ultralytics_adapter import (
    DISABLE_BUILTIN_AUGMENTATION,
    SharedAugmentationDetectionTrainer,
    build_train_dataset,
    write_ultralytics_yaml,
)
from ppe_detection_evaluation.experiment import DATA_ROOT, EPOCH_BUDGET, MODELS_DIR, PRETRAINED_DIR, SEED
from ppe_detection_evaluation.utilities.visualize import plot_examples

## Hyperparameters

Everything tunable for this model is set in the next cell. `SEED` and `EPOCH_BUDGET` are shared by all four notebooks
(`experiment.py`): the seed fixes the augmented examples and the budget caps every model at the same number of
epochs. Batch size and optimizer settings are chosen per model.

In [ ]:
# EPOCH_BUDGET is the shared scratch-run budget (10, experiment.py). For the actual runs, raise it there and
# pass patience=<N> to model.train (see the note at the top).
EPOCHS = EPOCH_BUDGET
BATCH_SIZE = 16   # Ultralytics' default; batch=-1 lets Ultralytics pick the largest that fits
# Optimizer, LR schedule and loss weights stay at Ultralytics' defaults (optimizer="auto").

In [ ]:
# SEED, DATA_ROOT and the model folders come from experiment.py and are shared by all four notebooks.
PROJECT_DIR = MODELS_DIR  # the run lands in models/<RUN_NAME>/ (weights/, curves, results.csv)
RUN_NAME = "yolo26m"
print(f"Epochs: {EPOCHS} | Batch: {BATCH_SIZE} | Image size: {IMAGE_SIZE}")

## 1. Dataset

[Ultralytics Construction-PPE](https://docs.ultralytics.com/datasets/detect/construction-ppe/): 11 classes, five
present/absent PPE pairs (helmet, vest, goggles, gloves, boots) plus `Person`. `download_dataset` is a no-op once
`data/construction-ppe` exists, so all four notebooks share one copy.

If more than ~15 % of `goggles` / `no_goggle` boxes (the smallest PPE) are under 16 px, raise the minimum of
`SCALE_RANGE` in `augmentations.py` from 0.75 to 0.85, for every model.

In [ ]:
download_dataset(DATA_ROOT)
splits = check_dataset(DATA_ROOT)  # raises if the class order differs from DEFAULT_CLASS_NAMES
print("Images per split:", splits)

print(f"\n{'Class':<10} {'Boxes':>6} {'<16 px':>7}")
for name, s in class_box_stats(DATA_ROOT, "train").items():
    print(f"{name:<10} {s['count']:>6} {s['under_16px']:>7.1%}")

# Ultralytics resolves relative paths in data.yaml against the CWD, so write one with absolute paths.
data_yaml = write_ultralytics_yaml(DATA_ROOT)

## 2. Training data

The training dataset below is built exactly as the trainer builds it, so the preview and the check show what the model
is actually fed. `seed=SEED` is also the shared augmentation seed (the dataset reads `args.seed`), and a callback
restarts the persistent DataLoader workers each epoch so they pick up the new epoch number.

In [ ]:
train_ds = build_train_dataset(data_yaml, seed=SEED)
print(f"Train: {len(train_ds)} images")

In [ ]:
# Four training images (columns) as the model sees them in epochs 0–2 (rows)
examples, titles = [], []
for epoch in range(3):
    train_ds.epoch = epoch
    for i in range(4):
        path, image, boxes, labels = train_ds.as_raw_example(i)
        examples.append((image, boxes, labels))
        titles.append(f"epoch {epoch} · {os.path.basename(path)[:20]}")
train_ds.epoch = 0
plot_examples(examples, titles);

**Check: same examples as the other models.** `verify_same_examples` pulls samples through this model's own
training dataset and asserts they equal the shared pipeline's output. The fingerprints must be identical in all four
training notebooks.

In [ ]:
augmentation = SharedAugmentation(seed=SEED)
fingerprints = {}
for epoch in (0, 1):
    train_ds.epoch = epoch
    fingerprints[epoch] = verify_same_examples(train_ds.as_raw_example, range(32), augmentation, epoch)
train_ds.epoch = 0
fingerprints

## 3. Model

COCO-pretrained YOLO26-M. When training starts, Ultralytics rebuilds the model with the dataset's 11 classes and
copies every pretrained tensor whose name and shape match. Only the last 1×1 conv of each of the three class branches
(and of their copies in the NMS-free one-to-one branch) starts fresh; the backbone, neck and box branch keep their COCO
weights.

In [ ]:
model = YOLO(PRETRAINED_DIR / "yolo26m.pt")  # COCO weights, downloaded here on first use
model.info()

## 4. Training

After training starts, open `train_batch0.jpg` in the run directory: it must show single images with mild scale,
colour and blur/noise changes, and **no** mosaic collages or grayscale frames.

In [ ]:
model.train(
    trainer=SharedAugmentationDetectionTrainer,
    data=str(data_yaml),
    epochs=EPOCHS,
    batch=BATCH_SIZE,
    imgsz=IMAGE_SIZE,
    seed=SEED,
    project=str(PROJECT_DIR),
    name=RUN_NAME,
    exist_ok=True,
    **DISABLE_BUILTIN_AUGMENTATION,
)

save_dir = model.trainer.save_dir
print(f"\nResults saved to: {save_dir}")

In [ ]:
with open(save_dir / "results.csv") as f:
    rows = list(csv.DictReader(f))
history = {key.strip(): [float(row[key]) for row in rows] for key in rows[0]}
epochs_x = history["epoch"]
best_epoch = epochs_x[max(range(len(rows)), key=lambda i: history["metrics/mAP50-95(B)"][i])]

# Loss terms differ per model (e.g. box/cls/l1 for YOLO26, giou/cls/l1 for RT-DETR), so read them from the CSV.
losses = [key.removeprefix("train/") for key in history if key.startswith("train/")]
fig, axes = plt.subplots(1, len(losses), figsize=(5 * len(losses), 4))
for ax, loss in zip(axes, losses):
    ax.plot(epochs_x, history[f"train/{loss}"], label="Train")
    ax.plot(epochs_x, history[f"val/{loss}"], label="Val")
    ax.axvline(best_epoch, color="grey", linestyle=":", lw=1.5, label=f"Best mAP@50-95 (epoch {best_epoch:.0f})")
    ax.set(xlabel="Epoch", ylabel="Loss", title=loss)
    ax.legend()
    ax.grid(True)
plt.suptitle("YOLO26-M (Ultralytics) — Training & validation loss", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(save_dir / "loss_curves.png", dpi=150)
plt.show()

In [ ]:
metrics = [
    ("metrics/precision(B)", "Precision"),
    ("metrics/recall(B)", "Recall"),
    ("metrics/mAP50(B)", "mAP@50"),
    ("metrics/mAP50-95(B)", "mAP@50-95"),
]
fig, axes = plt.subplots(1, len(metrics), figsize=(18, 4))
for ax, (key, label) in zip(axes, metrics):
    ax.plot(epochs_x, history[key], marker="o", markersize=3)
    ax.axvline(best_epoch, color="grey", linestyle=":", lw=1.5)
    ax.set(xlabel="Epoch", title=label)
    ax.grid(True)
plt.suptitle("YOLO26-M (Ultralytics) — Validation performance per epoch", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(save_dir / "val_metrics.png", dpi=150)
plt.show()

Next: evaluate `models/yolo26m/weights/best.pt` on the test split with [`yolo26m_eval.ipynb`](yolo26m_eval.ipynb).